# Parte 6 — Deployment
### Workshop: Clasificación de Emociones en Twitter

Este notebook despliega cualquier modelo del Hub como una API REST y muestra cómo consumirla.

**Este notebook es independiente** — no necesita los anteriores en la misma sesión.

## Estructura

```
┌─────────────────────────────────────────────────────┐
│  HF Hub  →  Predictor  →  FastAPI  →  /predict      │
│                                   →  /health        │
│                                   →  /models        │
└─────────────────────────────────────────────────────┘
```

Puedes apuntar el servidor a cualquiera de los tres modelos entrenados en el workshop:
- `your-username/tweeteval-emotion-distilbert`
- `your-username/tweeteval-emotion-bertweet`
- `your-username/tweeteval-emotion-bertweet-lora`

In [1]:
!pip install 'transformers[torch]' 'accelerate>=1.1.0' fastapi uvicorn python-multipart requests -q

## 1. Configuración

Cambia `HF_REPO` para apuntar al modelo que quieres desplegar.

In [2]:
# ── Elige el modelo a desplegar ───────────────────────────────────────────────
HF_REPO = "jezapataf/tweeteval-emotion-distilbert"   # <-- cambia esto

HOST    = "0.0.0.0"
PORT    = 8081
TOP_K   = 4      # número de predicciones a devolver (tenemos 4 clases)

LABEL_NAMES = ["anger", "joy", "optimism", "sadness"]

## 2. Predictor

La clase `Predictor` encapsula toda la lógica de inferencia. Acepta texto plano y devuelve las probabilidades de cada emoción.

In [3]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification


class Predictor:
    """Carga un modelo desde HF Hub y expone predict() para texto plano."""

    def __init__(self, hub_repo: str, top_k: int = 4, max_length: int = 128):
        self.device     = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.top_k      = top_k
        self.max_length = max_length
        self.hub_repo   = hub_repo

        print(f"Cargando modelo desde '{hub_repo}' ...")
        self.tokenizer = AutoTokenizer.from_pretrained(hub_repo)
        self.model     = AutoModelForSequenceClassification.from_pretrained(hub_repo)
        self.model.to(self.device).eval()

        self.id2label = self.model.config.id2label
        n_params = sum(p.numel() for p in self.model.parameters())
        print(f"Listo. Dispositivo: {self.device}  |  Parámetros: {n_params/1e6:.1f}M")

    @torch.inference_mode()
    def predict(self, text: str) -> list[dict]:
        """Devuelve top-k predicciones para un texto.

        Returns
        -------
        list of dicts con claves 'label' y 'score', ordenados por score descendente.
        """
        inputs = self.tokenizer(
            text,
            return_tensors="pt",
            truncation=True,
            max_length=self.max_length,
        ).to(self.device)

        logits = self.model(**inputs).logits
        probs  = torch.softmax(logits, dim=-1)[0]

        values, indices = probs.topk(min(self.top_k, len(self.id2label)))
        return [
            {"label": self.id2label[idx.item()], "score": round(val.item(), 6)}
            for val, idx in zip(values, indices)
        ]

    @torch.inference_mode()
    def predict_batch(self, texts: list[str]) -> list[list[dict]]:
        """Predice una lista de textos en un solo forward pass."""
        inputs = self.tokenizer(
            texts,
            return_tensors="pt",
            truncation=True,
            max_length=self.max_length,
            padding=True,
        ).to(self.device)

        probs = torch.softmax(self.model(**inputs).logits, dim=-1)

        results = []
        for row in probs:
            values, indices = row.topk(min(self.top_k, len(self.id2label)))
            results.append([
                {"label": self.id2label[idx.item()], "score": round(val.item(), 6)}
                for val, idx in zip(values, indices)
            ])
        return results


# Cargar el modelo una vez aquí — el servidor lo reutiliza
predictor = Predictor(HF_REPO, top_k=TOP_K)

Cargando modelo desde 'jezapataf/tweeteval-emotion-distilbert' ...


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Listo. Dispositivo: cuda  |  Parámetros: 67.0M


### Prueba rápida antes de levantar el servidor

In [4]:
tests = [
    "I'm so happy today!!! 😊",
    "this is absolutely disgusting #angry #wtf",
    "@user can't believe what just happened... 😢",
    "The future looks bright #hope #goals",
]

print(f"{'Texto':<50} {'Predicción':<12} {'Score':>6}")
print("-" * 72)
for text in tests:
    top = predictor.predict(text)[0]
    preview = (text[:47] + "...") if len(text) > 50 else text.ljust(50)
    print(f"{preview}  {top['label']:<12}  {top['score']:.4f}")

Texto                                              Predicción    Score
------------------------------------------------------------------------
I'm so happy today!!! 😊                             joy           0.9464
this is absolutely disgusting #angry #wtf           anger         0.9789
@user can't believe what just happened... 😢         sadness       0.8002
The future looks bright #hope #goals                optimism      0.8056


## 3. Servidor FastAPI

Tres endpoints:

| Método | Ruta | Descripción |
|---|---|---|
| `GET` | `/health` | Estado del servidor y modelo cargado |
| `GET` | `/model` | Información del modelo activo |
| `POST` | `/predict` | Predice la emoción de un texto |
| `POST` | `/predict/batch` | Predice una lista de textos |

In [5]:
from fastapi import FastAPI, HTTPException
from fastapi.responses import JSONResponse
from pydantic import BaseModel
import traceback

app = FastAPI(
    title="Emotion Classifier",
    description="Clasificación de emociones en tweets: anger, joy, optimism, sadness",
    version="1.0",
)


class TextInput(BaseModel):
    text: str


class BatchInput(BaseModel):
    texts: list[str]


@app.get("/health")
def health() -> dict:
    return {
        "status":       "ok",
        "model_loaded": predictor is not None,
        "device":       str(predictor.device),
    }


@app.get("/model")
def model_info() -> dict:
    n_params = sum(p.numel() for p in predictor.model.parameters())
    return {
        "hub_repo":   predictor.hub_repo,
        "labels":     list(predictor.id2label.values()),
        "parameters": f"{n_params/1e6:.1f}M",
        "device":     str(predictor.device),
    }


@app.post("/predict")
def predict(body: TextInput) -> JSONResponse:
    if not body.text.strip():
        raise HTTPException(status_code=400, detail="El campo 'text' no puede estar vacío")
    try:
        predictions = predictor.predict(body.text)
        return JSONResponse(content={
            "text":        body.text,
            "predictions": predictions,
        })
    except Exception as e:
        traceback.print_exc()
        raise HTTPException(status_code=500, detail=str(e))


@app.post("/predict/batch")
def predict_batch(body: BatchInput) -> JSONResponse:
    if not body.texts:
        raise HTTPException(status_code=400, detail="La lista 'texts' no puede estar vacía")
    if len(body.texts) > 64:
        raise HTTPException(status_code=400, detail="Máximo 64 textos por batch")
    try:
        results = predictor.predict_batch(body.texts)
        return JSONResponse(content={
            "results": [
                {"text": t, "predictions": p}
                for t, p in zip(body.texts, results)
            ]
        })
    except Exception as e:
        traceback.print_exc()
        raise HTTPException(status_code=500, detail=str(e))


print("App definida. Ejecuta la siguiente celda para levantar el servidor.")

App definida. Ejecuta la siguiente celda para levantar el servidor.


## 4. Levantar el servidor

Esta celda bloquea el kernel mientras el servidor está corriendo.  
Para detenerlo: **Kernel → Interrupt**.

In [6]:
%pip install nest_asyncio

Note: you may need to restart the kernel to use updated packages.


In [7]:
import asyncio
import uvicorn

HOST = "0.0.0.0"
PORT = 8000

config = uvicorn.Config(
    app,
    host=HOST,
    port=PORT,
    log_level="info"
)

server = uvicorn.Server(config)

print(f"Servidor en http://{HOST}:{PORT}")
print(f"Docs en      http://{HOST}:{PORT}/docs")

await server.serve()

INFO:     Started server process [33203]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


Servidor en http://0.0.0.0:8000
Docs en      http://0.0.0.0:8000/docs
INFO:     127.0.0.1:52866 - "GET /health HTTP/1.1" 200 OK
INFO:     127.0.0.1:52876 - "GET /model HTTP/1.1" 200 OK
INFO:     127.0.0.1:53402 - "POST /predict HTTP/1.1" 200 OK
INFO:     127.0.0.1:53412 - "POST /predict/batch HTTP/1.1" 200 OK


INFO:     Shutting down
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.
INFO:     Finished server process [33203]


## 7. Cargar un modelo diferente sin reiniciar

Para cambiar de modelo solo hay que reinstanciar el `Predictor` y volver a definir la app. No es necesario reiniciar el kernel.

In [8]:
# Descomenta el modelo que quieres usar y vuelve a ejecutar desde la celda del Predictor
# 7. Cargar un modelo diferente sin reiniciar

# Descomenta el modelo que quieres usar:

HF_REPO = "jezapataf/tweeteval-emotion-distilbert"
# HF_REPO = "jezapataf/tweeteval-emotion-bertweet"
# HF_REPO = "jezapataf/tweeteval-emotion-bertweet-lora"

predictor = Predictor(HF_REPO, top_k=TOP_K)

print(f"Modelo activo: {HF_REPO}")


Cargando modelo desde 'jezapataf/tweeteval-emotion-distilbert' ...


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Listo. Dispositivo: cuda  |  Parámetros: 67.0M
Modelo activo: jezapataf/tweeteval-emotion-distilbert


In [9]:
import asyncio
import uvicorn

HOST = "0.0.0.0"
PORT = 8000

config = uvicorn.Config(
    app,
    host=HOST,
    port=PORT,
    log_level="info"
)

server = uvicorn.Server(config)

print(f"Servidor en http://{HOST}:{PORT}")
print(f"Docs en      http://{HOST}:{PORT}/docs")

await server.serve()

INFO:     Started server process [33203]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


Servidor en http://0.0.0.0:8000
Docs en      http://0.0.0.0:8000/docs


INFO:     Shutting down
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.
INFO:     Finished server process [33203]
